# InterPro Fetch

This notebook demonstrates `run_interproscan_fetch`, which retrieves protein domain annotations from [InterPro](https://www.ebi.ac.uk/interpro/) — the EMBL-EBI aggregator covering Pfam, SMART, PROSITE, Gene3D / CATH-Gene3D, Panther, and the rest of the InterPro member-database catalog.

Two paths share the same Output schema: a fast **direct lookup** by UniProt accession (no submission required), and a **submit-and-scan** path against EBI's iprscan5 service for novel sequences (requires a contact email per EBI policy).

In [ ]:
from proto_tools.tools.database_retrieval import (
    InterProScanFetchConfig,
    InterProScanFetchInput,
    run_interproscan_fetch,
)

## Input API Reference

| Field | Type | Required | Description |
|---|---|---|---|
| `uniprot_id` | `str \| None` | one of | UniProt accession for direct entry lookup. |
| `sequence` | `str \| None` | one of | Protein sequence for submit-and-scan path. Requires `config.email`. |

Provide exactly one of `uniprot_id` or `sequence`.

## Config API Reference

| Field | Type | Default | Description |
|---|---|---|---|
| `email` | `str \| None` | `None` | Required by EBI for the sequence-submit path. |
| `applications` | `list[InterProApp] \| None` | `None` | Submit-only — restrict to specific member DBs. |
| `include_go_terms` | `bool` | `True` | Include GO term cross-refs in each row. |
| `include_pathways` | `bool` | `True` | Include pathway cross-refs (Reactome, MetaCyc). |

## Output API Reference

| Field | Type | Description |
|---|---|---|
| `accession` | `str \| None` | Resolved UniProt accession. |
| `sequence_length` | `int \| None` | Length of the queried protein. |
| `domains` | `list[InterProDomain]` | All hits across all member databases. |
| `num_domains` | `int` | `len(domains)`. |
| `job_id` | `str` | iprscan5 job ID (sequence path); empty for direct lookup. |
| `source_url` | `str` | InterPro entry URL or iprscan5 result URL. |
| `raw_entries` | `list[dict[str, Any]]` | Raw API JSON entries / matches. |

Each `InterProDomain` row carries `accession`, `name`, `type`, `member_database`, `integrated_ipr`, 1-indexed inclusive `start` / `end`, optional `score` / `model` / `representative`, and `go_terms` / `pathways` lists.

## Basic Usage

Fetch all InterPro hits for human TP53 (UniProt `P04637`) via the fast direct path. No contact email required because no job is submitted.

In [ ]:
output = run_interproscan_fetch(InterProScanFetchInput(uniprot_id="P04637"))
assert output.success
print(f"{output.num_domains} hits across {sorted({d.member_database for d in output.domains})}")

## Example: Find the canonical Pfam DNA-binding-domain anchor for TP53

In [ ]:
for domain in output.domains:
    if domain.member_database == "pfam" and domain.name.startswith("P53"):
        print(f"{domain.accession} {domain.name}: {domain.start}-{domain.end} ({domain.type})")

## Example: Build a `lock_mask` for "preserve the active site, redesign the rest"

Locks every residue covered by an `active_site`, `conserved_site`, `binding_site`, or `ptm` match. Use the resulting boolean mask as a per-residue constraint for any redesign tool.

In [ ]:
length = output.sequence_length or max(d.end for d in output.domains)
lock_mask = [False] * length
for domain in output.domains:
    if domain.type in {"active_site", "conserved_site", "binding_site", "ptm"}:
        for i in range(domain.start - 1, domain.end):
            lock_mask[i] = True
print(f"{sum(lock_mask)} of {length} residues locked")

## Submit-and-scan path (novel sequences)

When you have a sequence that is not in UniProt, submit it to iprscan5. **Set `config.email` to a real contact mailbox** — EBI requires it. Wall-clock varies (~2–30 min) with queue depth; the helper polls automatically.

```python
output = run_interproscan_fetch(
    InterProScanFetchInput(sequence="MKTI...your sequence..."),
    InterProScanFetchConfig(email="you@example.org"),
)
```